In [ ]:
import os
import glob
from crewai import Agent, Task, Crew, Process
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Set your API Key
os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"
gemini_llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

# --- THE FIX: FIND THE LATEST HEALTHY CHECKPOINT ---
checkpoint_dir = "../results/codet5-beginner-explainer"
checkpoints = glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))

if not checkpoints:
    print("Error: No checkpoints found. Did your training finish in this session?")
else:
    # Sort to find the highest step number (the most trained version)
    checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
    latest_ckpt = checkpoints[-1]


    # --- LOAD LOCAL CODET5 FROM THE CHECKPOINT ---
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(latest_ckpt)
    model = AutoModelForSeq2SeqLM.from_pretrained(latest_ckpt).to(device)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [20]:
# 1. Import BaseTool explicitly from CrewAI
from crewai.tools import BaseTool
from crewai import Agent, Task, Crew, Process

# --- THE BULLETPROOF TOOL CLASS ---
class LocalCodeT5Tool(BaseTool):
    name: str = "Local_CodeT5_Explainer"
    description: str = "Pass raw python code to this tool to get a technical baseline explanation."

    def _run(self, code_snippet: str) -> str:
        # We use your globally loaded model, tokenizer, and device here
        prompt = f"explain python: {code_snippet}"
        inputs = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=128)

        return tokenizer.decode(outputs[0], skip_special_tokens=True)

codet5_explain_tool = LocalCodeT5Tool()

In [21]:
# Phase 6: Personalization Variable
from crewai.tools import tool
user_level = "Beginner" # Options: "Beginner", "Intermediate"

# Explainer Agent: The technical lead
explainer_agent = Agent(
    role='Technical Code Analyst',
    goal='Generate a technically accurate baseline explanation of the provided code.',
    backstory='You specialize in understanding raw code logic. You use local specialized models to extract meaning.',
    tools=[codet5_explain_tool],
    llm="gemini/gemini-3.1-flash-lite",
    verbose=True
)

# Simplifier Agent: The educational expert (Phase 6 implementation)
simplifier_agent = Agent(
    role='Educational Personalization Expert',
    goal=f'Adapt technical explanations for a {user_level} programmer.',
    backstory=f'You are a CS professor who knows how to teach {user_level}s. You use analogies and simple terms.',
    llm="gemini/gemini-3.1-flash-lite",
    verbose=True
)

# Complexity Agent: The DSA specialist
complexity_agent = Agent(
    role='Algorithm Specialist',
    goal='Provide precise Big O Time and Space complexity analysis.',
    backstory='You live and breathe Data Structures and Algorithms. You provide concise complexity metrics.',
    llm="gemini/gemini-3.1-flash-lite",
    verbose=True
)

# Critic Agent: The Final Reviewer
critic_agent = Agent(
    role='Content Quality Lead',
    goal='Ensure the final explanation is cohesive, accurate, and perfectly formatted.',
    backstory='You ensure the output follows the required structure: Summary, Step-by-Step, Example, and Complexity.',
    llm="gemini/gemini-3.1-flash-lite",
    verbose=True
)

In [22]:
code_to_explain = """
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
"""

task1 = Task(
    description=f"Analyze this code using the Local_CodeT5_Explainer tool: {code_to_explain}",
    agent=explainer_agent,
    expected_output="A technical summary of the code logic."
)

task2 = Task(
    description=f"Take the technical summary and simplify it for a {user_level} user. Use analogies if helpful.",
    agent=simplifier_agent,
    expected_output=f"A {user_level}-friendly, step-by-step guide."
)

task3 = Task(
    description=f"Determine the Time and Space complexity for: {code_to_explain}",
    agent=complexity_agent,
    expected_output="Time: O(n), Space: O(n) for recursive stack."
)

task4 = Task(
    description="Combine all previous outputs into a structured Markdown report.",
    agent=critic_agent,
    expected_output="The final finalized personalized code explanation report.",
    context=[task1, task2, task3]
)

# Phase 5: Multi-Agent System Assembly
explanation_crew = Crew(
    agents=[explainer_agent, simplifier_agent, complexity_agent, critic_agent],
    tasks=[task1, task2, task3, task4],
    process=Process.sequential
)

result = explanation_crew.kickoff()
print("\n\n########################")
print("## FINAL EXPLANATION ##")
print("########################\n")
print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Code Analyst                                                                                  │
│                                                                                                                 │
│  Task: Analyze this code using the Local_CodeT5_Explainer tool:                                                 │
│  def factorial(n):                                                                                              │
│      if n == 0:                                                                                                 │
│          return 1                                                                                               │
│      else:                                                                                                      │
│          return n * factorial(n-1)                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool local_code_t5_explainer executed with result: This code calculates factorial function. It uses a factorial function to get the factorial value. It returns the factorial value, then returns the factorial value....


ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x785f9a9785c0>: Failed to establish a new connection: [Errno 111] Connection refused'))


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Code Analyst                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The provided code defines a recursive function named `factorial` that computes the factorial of a              │
│  non-negative integer `n`. The logic follows a standard recursive definition: it establishes a base case where  │
│  `if n == 0`, the function returns `1` (as 0! is defined as 1). In the recursive step, the function returns     │
│  the product of `n` and the result of a recursive call to `factorial(n-1)`, effectively implementing the        │
│  mathematical formula n! = n * (n-1)!.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Educational Personalization Expert                                                                      │
│                                                                                                                 │
│  Task: Take the technical summary and simplify it for a Beginner user. Use analogies if helpful.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Quality Lead                                                                                    │
│                                                                                                                 │
│  Task: Combine all previous outputs into a structured Markdown report.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x785f8861b080>: Failed to establish a new connection: [Errno 111] Connection refused'))


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Quality Lead                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Understanding Recursion: The Factorial Function                                                              │
│                                                                                                                 │
│  ## Summary                                                                                                     │
│  The provided code defines a recursive function named `factorial` that computes the factorial of a              │
│  non-negative integer `n`. The logic follows the mathematical definition $n! = n \times (n-1)!$. It employs a   │
│  base case to terminate recursion at $0$ and a recursive step to repeatedly reduce the problem size until that  │
│  base case is reached.                                                                                          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Step-by-Step                                                                                                │
│  1.  **Define the Base Case:** Every recursive function requires a stopping condition to prevent infinite       │
│  execution. Here, `if n == 0: return 1` handles the mathematical definition that $0! = 1$.                      │
│  2.  **Define the Recursive Step:** For any $n > 0$, the function calls itself with a smaller input: `n - 1`.   │
│  3.  **Perform the Operation:** The function calculates the current value of `n` multiplied by the result of    │
│  the recursive call `factorial(n - 1)`.                                                                         │
│  4.  **Unwind the Stack:** Once the base case is hit, the results are passed back up through the chain of       │
│  function calls, multiplying each previous `n` value until the final product is returned.                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Example                                                                                                     │
│  To visualize the calculation of `factorial(3)`, consider the following sequence:                               │
│                                                                                                                 │
│  *   **Call 1:** `factorial(3)` calls `3 * factorial(2)`                                                        │
│  *   **Call 2:** `factorial(2)` calls `2 * factorial(1)`                                                        │
│  *   **Call 3:** `factorial(1)` calls `1 * factorial(0)`                                                        │
│  *   **Call 4 (Base Case):** `factorial(0)` returns `1`                                                         │
│  *   **Resolution:**                                                                                            │
│      *   `factorial(1)` returns `1 * 1 = 1`            



########################
## FINAL EXPLANATION ##
########################

# Understanding Recursion: The Factorial Function

## Summary
The provided code defines a recursive function named `factorial` that computes the factorial of a non-negative integer `n`. The logic follows the mathematical definition $n! = n \times (n-1)!$. It employs a base case to terminate recursion at $0$ and a recursive step to repeatedly reduce the problem size until that base case is reached.

---

## Step-by-Step
1.  **Define the Base Case:** Every recursive function requires a stopping condition to prevent infinite execution. Here, `if n == 0: return 1` handles the mathematical definition that $0! = 1$.
2.  **Define the Recursive Step:** For any $n > 0$, the function calls itself with a smaller input: `n - 1`.
3.  **Perform the Operation:** The function calculates the current value of `n` multiplied by the result of the recursive call `factorial(n - 1)`.
4.  **Unwind the Stack:** Once the base case is h

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯